# Customer Churn Prediction: Profit-Based Evaluation of ML Models
## Step 2: Data Cleaning & Preparation

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

# Load raw data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Raw data loaded: {df.shape[0]} rows × {df.shape[1]} columns ✓')

### 2.1 — Drop Irrelevant Columns

In [ ]:
# customerID is a unique identifier — not useful for modelling
df.drop(columns=['customerID'], inplace=True)
print('Dropped customerID column ✓')
print(f'Shape now: {df.shape}')

### 2.2 — Fix TotalCharges (hidden spaces → NaN → fill)

In [ ]:
# TotalCharges is object type with whitespace strings for new customers
print('Before fix — TotalCharges dtype:', df['TotalCharges'].dtype)

# Replace whitespace strings with NaN, then convert to float
df['TotalCharges'] = df['TotalCharges'].str.strip()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('After conversion — TotalCharges dtype:', df['TotalCharges'].dtype)

# Check how many NaNs were created
n_missing = df['TotalCharges'].isnull().sum()
print(f'NaN values after conversion: {n_missing}')

In [ ]:
# These NaN rows are brand-new customers (tenure = 0)
# Their TotalCharges should logically be 0.0
print('Rows with NaN TotalCharges:')
print(df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']].head(10))

# Fill with 0 — logical since they haven't been charged yet
df['TotalCharges'].fillna(0.0, inplace=True)
print(f'\nFilled {n_missing} NaN values with 0.0 ✓')

### 2.3 — Check for Duplicates

In [ ]:
duplicates = df.duplicated().sum()
print(f'Duplicate rows found: {duplicates}')

if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f'Removed {duplicates} duplicate rows ✓')
else:
    print('No duplicates — dataset is clean ✓')

### 2.4 — Check Remaining Missing Values

In [ ]:
print('=== Missing Values After Cleaning ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values remaining ✓')

### 2.5 — Fix Data Types & Encode Target Variable

In [ ]:
# SeniorCitizen is already 0/1 integer — confirm
print('SeniorCitizen unique values:', df['SeniorCitizen'].unique())

# Encode target: Churn → 1 (Yes), 0 (No)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print('Churn encoded: Yes → 1, No → 0 ✓')
print('Churn value counts:\n', df['Churn'].value_counts())

### 2.6 — Standardise Categorical Values

In [ ]:
# Some columns have 'No internet service' or 'No phone service'
# These are functionally equivalent to 'No' — simplify them

cols_to_simplify = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

for col in cols_to_simplify:
    df[col] = df[col].replace({
        'No internet service': 'No',
        'No phone service': 'No'
    })

print('Simplified redundant categories ✓')
print('\nSample check — MultipleLines:', df['MultipleLines'].unique())
print('Sample check — OnlineSecurity:', df['OnlineSecurity'].unique())

### 2.7 — Outlier Check on Numeric Columns

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='steelblue', color='navy'),
                    medianprops=dict(color='white', linewidth=2))
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_ylabel('Value')

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
    print(f'{col}: {len(outliers)} outliers detected (IQR method)')

plt.suptitle('Outlier Check — Numeric Columns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_03_outliers.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNote: No extreme outliers requiring removal — values are natural customer variation ✓')

### 2.8 — Final Cleaned Dataset Overview

In [ ]:
print('=== Cleaned Dataset — Data Types ===')
print(df.dtypes)
print(f'\nFinal shape: {df.shape}')

In [ ]:
print('=== Cleaned Dataset — First 5 Rows ===')
df.head()

In [ ]:
# Save the cleaned dataframe for use in all future steps
df.to_csv('churn_cleaned.csv', index=False)
print('Cleaned dataset saved as churn_cleaned.csv ✓')

print('\n' + '='*50)
print('       CLEANING SUMMARY')
print('='*50)
print('  ✓ Dropped customerID (irrelevant)')
print('  ✓ Fixed TotalCharges (string → float, NaN → 0)')
print('  ✓ Checked & confirmed no duplicates')
print('  ✓ Encoded Churn target (Yes/No → 1/0)')
print('  ✓ Simplified redundant category labels')
print('  ✓ Outlier check — no removal needed')
print('  ✓ Saved: churn_cleaned.csv')
print('='*50)
print('Step 2 Complete ✓  →  Proceed to Step 3: EDA & Visualization')